In [10]:
import requests

tiny_shakespeare_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# with open(tiny_shakespeare) as f:
#     text = f.read()

res = requests.get(tiny_shakespeare_url)
res.raise_for_status()

text = res.text
text[:10]

'First Citi'

In [11]:
# Length of the dataset
len(text)

1115394

In [12]:
# Building the vocabulary
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
print(''.join(vocab))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [16]:
# Create mapping for string to integers and vice-versa
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for i,ch in enumerate(vocab)}

# Create encoder and decoder
# Encoder
encoder = lambda s: [stoi[c] for c in s] # string -> list of int
decoder = lambda l: ''.join([itos[i] for i in l]) # list of int -> str

print(encoder('hi there'))
print(decoder(encoder('hi there')))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [21]:
import torch
data = torch.tensor(encoder(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [23]:
# Split the dataset into train and validation set
n = int(len(data)*0.9)
train_data = data[:n]
val_data = data[n:]

In [24]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [43]:
x = train_data[: block_size]
y = train_data[1: block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is: {context} Then output is : {target}")

When input is: tensor([18]) Then output is : 47
When input is: tensor([18, 47]) Then output is : 56
When input is: tensor([18, 47, 56]) Then output is : 57
When input is: tensor([18, 47, 56, 57]) Then output is : 58
When input is: tensor([18, 47, 56, 57, 58]) Then output is : 1
When input is: tensor([18, 47, 56, 57, 58,  1]) Then output is : 15
When input is: tensor([18, 47, 56, 57, 58,  1, 15]) Then output is : 47
When input is: tensor([18, 47, 56, 57, 58,  1, 15, 47]) Then output is : 58


In [75]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process parallely?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(train_data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('Input:')
print(xb.shape)
print(xb)
print('Output:')
print(yb.shape)
print(yb)

print('------')

for b in range(batch_size): # batch dimension
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is: {context.tolist()} Then output is : {target}")

Input:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Output:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------
When input is: [24] Then output is : 43
When input is: [24, 43] Then output is : 58
When input is: [24, 43, 58] Then output is : 5
When input is: [24, 43, 58, 5] Then output is : 57
When input is: [24, 43, 58, 5, 57] Then output is : 1
When input is: [24, 43, 58, 5, 57, 1] Then output is : 46
When input is: [24, 43, 58, 5, 57, 1, 46] Then output is : 43
When input is: [24, 43, 58, 5, 57, 1, 46, 43] Then output is : 39
When input is: [44] Then output is : 53
When input is: [44, 53] Then output is : 56
When input is: [44, 53, 56] Then output is : 1
When input is: [44, 53, 56, 1] Then output is : 5